# Ball Perception

Utilizing YOLO we test inference for ball bounding as part of our IRB140 perception pipeline.

**Phase 1 (this notebook):** confirm a COCO-pretrained Ultralytics YOLO model classifies a ball in
a scene, using `test_image.png` in this directory — which is a **Gazebo sim render** of the
`my_sphere` red sphere on a near-neutral background.

COCO includes class **32 `sports ball`** (soccer / tennis / basketball / … collapsed into one
generic class), so YOLO gives us a bounding box with **no training**. On this sim frame
`yolov8n.pt` reaches ~0.72 confidence and is stable across thresholds; `yolo11n.pt` only reaches
~0.48, and `yolo11s.pt` misses the ball entirely (a bigger COCO model is *not* automatically better
on out-of-distribution sim renders). See the sweep cell. It does not distinguish ball sub-types,
and small / blurred / occluded balls are the weak spot — see the fine-tune plan at the bottom.

The last section sketches the depth → 3D → TF step the runtime ROS node will do (the node itself
is a separate task; `test_image.png` has no depth channel so that part is illustrative here).

In [ ]:
%matplotlib inline
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import ultralytics
from ultralytics import YOLO

print("ultralytics", ultralytics.__version__)
print("torch      ", torch.__version__, "| CUDA:", torch.cuda.is_available())

In [ ]:
IMG_PATH = Path("test_image.png")
# yolov8n scores this sim frame ~0.72 vs ~0.48 for yolo11n (see the sweep cell); yolo11s misses it
# entirely. Same "nano" speed tier, so no runtime cost to preferring yolov8n.
MODEL_NAME = "yolov8n.pt"       # COCO-pretrained; auto-downloads on first use
SPORTS_BALL_ID = 32             # COCO class id for 'sports ball'
# One ball, plain background, nothing else ball-shaped -> keep the threshold low and just take the
# top-scoring 'sports ball' box. conf is only a cutoff on returned boxes, not a quality dial; a low
# value buys margin on harder future frames (gripper in view, ball at range, motion blur).
CONF = 0.15                     # detection confidence threshold
DEVICE = 0 if torch.cuda.is_available() else "cpu"

assert IMG_PATH.is_file(), f"missing test image: {IMG_PATH.resolve()}"

In [ ]:
model = YOLO(MODEL_NAME)

# sanity check: the class we care about exists in these weights
print("class", SPORTS_BALL_ID, "->", model.names[SPORTS_BALL_ID])
assert model.names[SPORTS_BALL_ID] == "sports ball"

In [ ]:
# cv2 loads BGR and drops the alpha channel (test_image.png is RGBA)
img_bgr = cv2.imread(str(IMG_PATH), cv2.IMREAD_COLOR)
assert img_bgr is not None, f"cv2 failed to read {IMG_PATH}"
H, W = img_bgr.shape[:2]
print(f"image: {W}x{H}")

plt.figure(figsize=(8, 6))
plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
plt.title("test_image.png")
plt.axis("off")
plt.show()

In [ ]:
result = model.predict(img_bgr, conf=CONF, device=DEVICE, verbose=False)[0]
print(f"{len(result.boxes)} detection(s) at conf >= {CONF}")

In [ ]:
ball_boxes = []  # (x1, y1, x2, y2, conf)
for b in result.boxes:
    cls_id = int(b.cls)
    conf = float(b.conf)
    x1, y1, x2, y2 = (float(v) for v in b.xyxy[0].tolist())
    print(f"  {model.names[cls_id]:<15} conf={conf:.3f}  bbox=({x1:.0f}, {y1:.0f}, {x2:.0f}, {y2:.0f})")
    if cls_id == SPORTS_BALL_ID:
        ball_boxes.append((x1, y1, x2, y2, conf))

if ball_boxes:
    # downstream contract: one ball in the scene -> the node takes the single top-scoring box
    best_box = max(ball_boxes, key=lambda b: b[4])
    bx1, by1, bx2, by2, bconf = best_box
    bu, bv = 0.5 * (bx1 + bx2), 0.5 * (by1 + by2)
    print(f"\nPASS - {len(ball_boxes)} sports-ball detection(s), best conf = {bconf:.3f}")
    print(f"       node would use: bbox=({bx1:.0f}, {by1:.0f}, {bx2:.0f}, {by2:.0f})  centre=({bu:.0f}, {bv:.0f})")
else:
    best_box = None
    print("\nFAIL - no 'sports ball' detected. Lower CONF, try a bigger model in the sweep below,")
    print("       or run the fine-tune plan at the bottom of this notebook.")

In [ ]:
# left: everything YOLO found; right: only the 'sports ball' boxes
ball_only = img_bgr.copy()
for x1, y1, x2, y2, conf in ball_boxes:
    cv2.rectangle(ball_only, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 3)
    cv2.putText(ball_only, f"sports ball {conf:.2f}", (int(x1), int(y1) - 8),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

fig, ax = plt.subplots(1, 2, figsize=(15, 6))
ax[0].imshow(cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB))
ax[0].set_title("result.plot() - all classes")
ax[1].imshow(cv2.cvtColor(ball_only, cv2.COLOR_BGR2RGB))
ax[1].set_title(f"sports ball only ({len(ball_boxes)})")
for a in ax:
    a.axis("off")
plt.show()

## Classical cross-check — HSV red mask + enclosing circle

COCO YOLO at ~0.72 is workable but not strong, and it will degrade on the live eye-in-hand stream.
This scene is so clean that a colour threshold is a near-perfect detector on its own. Running both
and checking their centres agree is a cheap confidence signal; the circle centre is also a better
(sub-pixel, symmetric) point to back-project than the YOLO box centre.

In [ ]:
# Classical fallback / cross-check: this palette (saturated red sphere on a neutral ground) is
# trivially segmentable, so an HSV mask + minEnclosingCircle is a near-100%-reliable detector
# (~1 ms, no GPU, sub-pixel centre). The ROS node can run this alongside YOLO and reconcile the two.
hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
mask = cv2.inRange(hsv, (0, 120, 70), (10, 255, 255)) | cv2.inRange(hsv, (170, 120, 70), (180, 255, 255))
k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k)
mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k)

contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
hsv_vis = img_bgr.copy()
if contours:
    (hx, hy), hr = cv2.minEnclosingCircle(max(contours, key=cv2.contourArea))
    cv2.circle(hsv_vis, (int(hx), int(hy)), int(hr), (255, 0, 0), 3)
    cv2.drawMarker(hsv_vis, (int(hx), int(hy)), (255, 0, 0), cv2.MARKER_CROSS, 20, 2)
    print(f"HSV circle: centre=({hx:.1f}, {hy:.1f})  radius={hr:.1f}px")
    if best_box is not None:
        d = float(np.hypot(hx - bu, hy - bv))
        print(f"YOLO box centre=({bu:.1f}, {bv:.1f})  ->  HSV vs YOLO centre distance = {d:.1f}px")
else:
    print("HSV: no red blob found (tune the hue/sat/val bounds for your lighting)")

fig, ax = plt.subplots(1, 2, figsize=(15, 6))
ax[0].imshow(mask, cmap="gray")
ax[0].set_title("HSV red mask")
ax[1].imshow(cv2.cvtColor(hsv_vis, cv2.COLOR_BGR2RGB))
ax[1].set_title("minEnclosingCircle")
for a in ax:
    a.axis("off")
plt.show()

In [ ]:
# Quick robustness sweep: model size x confidence threshold.
# Bigger models auto-download on first use (needs internet); failures are skipped.
sweep_models = ["yolo11n.pt", "yolo11s.pt", "yolov8n.pt"]
sweep_conf = [0.10, 0.25, 0.50]

print(f"{'model':<12} {'conf':>5} {'#ball':>6} {'best_conf':>10}")
print("-" * 38)
for mname in sweep_models:
    try:
        m = YOLO(mname)
    except Exception as e:  # noqa: BLE001 - download/load failure, keep sweeping
        print(f"{mname:<12}  (skipped: {e})")
        continue
    for c in sweep_conf:
        r = m.predict(img_bgr, conf=c, device=DEVICE, verbose=False)[0]
        confs = [float(b.conf) for b in r.boxes if int(b.cls) == SPORTS_BALL_ID]
        best = f"{max(confs):.3f}" if confs else "-"
        print(f"{mname:<12} {c:>5.2f} {len(confs):>6} {best:>10}")

## Depth → 3D → TF (what the runtime node will do)

The ROS node (`ball_detector.py`, a later task — modelled on `hybraut_irb140/lego_detector.py`) will:

1. Run this same YOLO inference on `/camera/color/image_raw`, take the `sports ball` box centre `(u, v)`.
2. Sample the **aligned** depth image (sim publishes `32FC1` metres on `/camera/depth/image_rect_raw`;
   a real RealSense D405 publishes `16UC1` mm — `perception_geometry.depth_to_meters()` handles both)
   over a small median window at `(u, v)` to get range `z`.
3. Back-project with the pinhole model (`image_geometry.PinholeCameraModel` from `/camera/color/camera_info`):
   `X = (u - cx) * z / fx`, `Y = (v - cy) * z / fy`, `Z = z` — a point in `depth_camera_optical`.
   This is exactly `perception_geometry.backproject_pixel()`.
4. tf2-transform that point `depth_camera_optical` → `base_link` (same retry-on-extrapolation pattern
   as `lego_detector._transform_point`).
5. Broadcast `base_link` → `ball` as a `geometry_msgs/TransformStamped` via `tf2_ros.TransformBroadcaster`.

The cell below just illustrates step 3 on the detected box centre with a **placeholder** `z` —
`test_image.png` carries no depth, and its intrinsics are unknown, so we borrow the sim camera's.

In [ ]:
if best_box is None:
    print("no ball box - skipping the deprojection demo")
else:
    x1, y1, x2, y2, _ = best_box
    u_img, v_img = 0.5 * (x1 + x2), 0.5 * (y1 + y2)

    # Sim RGBD camera: 640x480, horizontal FOV 1.3962634 rad (from the xacro sensor def).
    SIM_W, SIM_H, HFOV = 640, 480, 1.3962634
    fx = fy = (SIM_W / 2.0) / np.tan(HFOV / 2.0)
    cx, cy = SIM_W / 2.0, SIM_H / 2.0

    # rescale the box centre from the test image onto the sim grid (illustrative only)
    u = u_img * SIM_W / W
    v = v_img * SIM_H / H

    z = 0.50  # PLACEHOLDER metres - at runtime this comes from the aligned depth image
    X = (u - cx) * z / fx
    Y = (v - cy) * z / fy
    Z = z
    print(f"box centre (sim px): u={u:.1f}, v={v:.1f}")
    print(f"fx=fy={fx:.1f}  cx={cx:.0f}  cy={cy:.0f}")
    print(f"ball in depth_camera_optical (z={z} m placeholder): "
          f"X={X:+.3f}  Y={Y:+.3f}  Z={Z:+.3f}  [m]")
    print("\n(at runtime: replace z with the median depth sample, then tf2 -> base_link, then broadcast TF)")

## Fine-tune plan (if COCO `sports ball` confidence is not enough)

**Trigger:** `test_image.png` is already a sim frame and COCO `yolov8n` only reaches ~0.72 on it. Run
this notebook against live `/camera/color/image_raw` frames — if confidence sags with the real
eye-in-hand viewpoint (gripper occlusion, the 0.03 m sphere at working distance, motion), fine-tune a
single-class model:

1. **Capture** frames: launch the sim (`ros2 launch abb_irb140_description gazebo.launch.py`), record
   `/camera/color/image_raw` (rosbag or a small subscriber that dumps PNGs), moving the arm to vary
   viewpoint / range / lighting. Aim for a few hundred frames. (The HSV detector above can auto-label
   most of these — export its circle as a YOLO box.)
2. **Label** the ball boxes — Roboflow, CVAT, or Label Studio — and export **YOLO format**
   (`images/`, `labels/`, `data.yaml` with a single class `ball`).
3. **Train:** `YOLO("yolov8n.pt").train(data="ball.yaml", epochs=100, imgsz=640, device=0)`.
4. **Deploy:** copy `runs/detect/train/weights/best.pt` to
   `perception/ball_perception/weights/ball_yolo.pt` (mirrors the lego weights convention); the
   future `ball_detector.py` node loads that as its default `model_path`.

A single-class custom model also removes the COCO `sports ball` ambiguity and is faster.